# 04 - Modeling

This notebook trains, evaluates, and compares baseline and behavior-aware machine learning models for customer behavior shift detection.

In [23]:
import pandas as pd
import numpy as np
from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score, average_precision_score)

In [14]:
# Load the modeling dataset
file_path = "../data/processed/behavior_change_dataset.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

df.head()

Dataset shape: (19651, 32)
Columns:
['Customer ID', 'Month', 'transaction_count', 'total_quantity', 'total_spending', 'average_transaction_value', 'unique_products', 'previous_month', 'months_since_previous', 'previous_transaction_count', 'previous_total_quantity', 'previous_total_spending', 'previous_average_transaction_value', 'previous_unique_products', 'change_transaction_count', 'change_total_quantity', 'change_total_spending', 'change_average_transaction_value', 'change_unique_products', 'pct_change_transaction_count', 'pct_change_total_quantity', 'pct_change_total_spending', 'pct_change_average_transaction_value', 'pct_change_unique_products', 'large_change_count', 'behavior_shift_candidate', 'core_large_change_count', 'core_behavior_shift_candidate', 'behavior_shift', 'historical_active_months', 'historical_transactions', 'historical_spending']


,Customer ID,Month,transaction_count,total_quantity,total_spending,average_transaction_value,unique_products,previous_month,months_since_previous,previous_transaction_count,...,pct_change_average_transaction_value,pct_change_unique_products,large_change_count,behavior_shift_candidate,core_large_change_count,core_behavior_shift_candidate,behavior_shift,historical_active_months,historical_transactions,historical_spending
0,12346.0,2010-06,1,19,142.31,7.490000,19,2010-03,3,1.0,...,3.844732e+01,280.000000,3,True,3,True,1,0,1.0,27.05
1,12346.0,2011-01,1,74215,77183.60,77183.600000,1,2010-06,7,1.0,...,1.030389e+06,-94.736842,3,True,3,True,1,1,2.0,169.36
2,12347.0,2010-12,1,319,711.79,22.960968,31,2010-10,2,1.0,...,5.018702e+01,-22.500000,0,False,0,False,0,0,1.0,611.53
3,12347.0,2011-01,1,315,475.39,16.392759,29,2010-12,1,1.0,...,-2.860598e+01,-6.451613,0,False,0,False,0,1,2.0,1323.32
4,12347.0,2011-04,1,483,636.25,26.510417,24,2011-01,3,1.0,...,6.172029e+01,-17.241379,0,False,0,False,0,2,3.0,1798.71


In [15]:
# Define target
target_column = "behavior_shift"

# Baseline features
baseline_features = [
    "historical_active_months",
    "historical_transactions",
    "historical_spending"
]

# Behavior-aware features
behavior_aware_features = [
    "historical_active_months",
    "historical_transactions",
    "historical_spending",
    "previous_transaction_count",
    "previous_total_quantity",
    "previous_total_spending",
    "previous_average_transaction_value",
    "previous_unique_products",
    "months_since_previous"
]

print("Target:", target_column)

print("\nBaseline features:")
print(baseline_features)

print("\nBehavior-aware features:")
print(behavior_aware_features)

Target: behavior_shift

Baseline features:
['historical_active_months', 'historical_transactions', 'historical_spending']

Behavior-aware features:
['historical_active_months', 'historical_transactions', 'historical_spending', 'previous_transaction_count', 'previous_total_quantity', 'previous_total_spending', 'previous_average_transaction_value', 'previous_unique_products', 'months_since_previous']


In [16]:
# Define time-based split boundaries

train_end = "2011-05"
validation_end = "2011-08"

# Training set
train_df = df[
    df["Month"] <= train_end
].copy()

# Validation set
validation_df = df[
    (df["Month"] > train_end) &
    (df["Month"] <= validation_end)
].copy()

# Test set
test_df = df[
    df["Month"] > validation_end
].copy()

print("Train period:",
      train_df["Month"].min(), "to", train_df["Month"].max())

print("Validation period:",
      validation_df["Month"].min(), "to", validation_df["Month"].max())

print("Test period:",
      test_df["Month"].min(), "to", test_df["Month"].max())

print("\nRows:")
print("Train:", len(train_df))
print("Validation:", len(validation_df))
print("Test:", len(test_df))

Train period: 2010-01 to 2011-05
Validation period: 2011-06 to 2011-08
Test period: 2011-09 to 2011-12

Rows:
Train: 12832
Validation: 2553
Test: 4266


In [24]:
# Check target distribution across all data splits

for name, dataset in [
    ("Train", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:
    print(f"\n{name} target distribution:")
    
    print(dataset[target_column].value_counts().sort_index())
    
    print("\nPercentage:")
    print(
        dataset[target_column]
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    )


Train target distribution:
behavior_shift
0    10745
1     2087
Name: count, dtype: int64

Percentage:
behavior_shift
0    83.74
1    16.26
Name: proportion, dtype: float64

Validation target distribution:
behavior_shift
0    2198
1     355
Name: count, dtype: int64

Percentage:
behavior_shift
0    86.09
1    13.91
Name: proportion, dtype: float64

Test target distribution:
behavior_shift
0    3471
1     795
Name: count, dtype: int64

Percentage:
behavior_shift
0    81.36
1    18.64
Name: proportion, dtype: float64


In [18]:
# Prepare baseline features and target

X_train_baseline = train_df[baseline_features].copy()
X_validation_baseline = validation_df[baseline_features].copy()
X_test_baseline = test_df[baseline_features].copy()

y_train = train_df[target_column].copy()
y_validation = validation_df[target_column].copy()
y_test = test_df[target_column].copy()

print("Baseline training shape:", X_train_baseline.shape)
print("Baseline validation shape:", X_validation_baseline.shape)
print("Baseline test shape:", X_test_baseline.shape)

print("\nTarget shapes:")
print("Train:", y_train.shape)
print("Validation:", y_validation.shape)
print("Test:", y_test.shape)

Baseline training shape: (12832, 3)
Baseline validation shape: (2553, 3)
Baseline test shape: (4266, 3)

Target shapes:
Train: (12832,)
Validation: (2553,)
Test: (4266,)


In [19]:
# Define baseline preprocessing

baseline_log_features = [
    "historical_transactions",
    "historical_spending"
]

baseline_numeric_features = [
    "historical_active_months"
]

baseline_preprocessor = ColumnTransformer(
    transformers=[
        (
            "log",
            Pipeline([
                ("log1p", FunctionTransformer(np.log1p)),
                ("scaler", StandardScaler())
            ]),
            baseline_log_features
        ),
        (
            "numeric",
            StandardScaler(),
            baseline_numeric_features
        )
    ]
)

print("Baseline preprocessing pipeline created.")

Baseline preprocessing pipeline created.


In [20]:
# Build the baseline Logistic Regression pipeline

baseline_model = Pipeline([
    ("preprocessing", baseline_preprocessor),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

print("Baseline Logistic Regression pipeline created.")

Baseline Logistic Regression pipeline created.


In [21]:
# Fit the baseline model on the training data

baseline_model.fit(
    X_train_baseline,
    y_train
)

print("Baseline model trained successfully.")

Baseline model trained successfully.


In [22]:
# Generate predictions on the validation set

y_validation_pred = baseline_model.predict(
    X_validation_baseline
)

y_validation_proba = baseline_model.predict_proba(
    X_validation_baseline
)[:, 1]

print("Validation predictions generated.")
print("Predictions shape:", y_validation_pred.shape)
print("Probabilities shape:", y_validation_proba.shape)

Validation predictions generated.
Predictions shape: (2553,)
Probabilities shape: (2553,)


In [25]:
baseline_validation_metrics = {
    "Precision": precision_score(y_validation, y_validation_pred),
    "Recall": recall_score(y_validation, y_validation_pred),
    "F1": f1_score(y_validation, y_validation_pred),
    "ROC-AUC": roc_auc_score(y_validation, y_validation_proba),
    "PR-AUC": average_precision_score(
        y_validation,
        y_validation_proba
    )
}

print("Baseline Validation Metrics:")
for metric, value in baseline_validation_metrics.items():
    print(f"{metric}: {value:.4f}")

Baseline Validation Metrics:
Precision: 0.0000
Recall: 0.0000
F1: 0.0000
ROC-AUC: 0.5763
PR-AUC: 0.1804


## Baseline Validation Probability Analysis

The predicted probabilities are inspected to understand how the baseline model
separates the two target classes.

We compare the probability distribution overall and the average predicted
probability for each actual class.

In [27]:
# Inspect the distribution of predicted probabilities and compare them by actual class

print("Validation probability summary:")

print(pd.Series(y_validation_proba).describe())

print("\nMean probability by actual class:")

print(
    pd.DataFrame({
        "target": y_validation.values,
        "probability": y_validation_proba
    })
    .groupby("target")["probability"]
    .agg(["count", "mean", "median", "min", "max"])
)

Validation probability summary:
count    2553.000000
mean        0.182089
std         0.054922
min         0.040110
25%         0.144876
50%         0.175919
75%         0.213064
max         0.572563
dtype: float64

Mean probability by actual class:
        count      mean    median       min       max
target                                               
0        2198  0.180073  0.174376  0.040110  0.572563
1         355  0.194573  0.194415  0.053993  0.456900
